# Step 4: Credit Card Fraud Exploratory Data Analysis

This notebook analyzes the original `data/raw/creditcard.csv` dataset. It preserves all transaction rows, including every fraudulent transaction. No cleaning, resampling, or model training is performed.

Each chart is followed by a short interpretation. Statistical findings below are calculated from the loaded dataset at execution time.

## 1. Import Libraries and Load Dataset

Load the original CSV directly. The notebook only reads this file; all derived columns and calculations stay in memory.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots
from IPython.display import display

DATASET_RELATIVE_PATH = Path("data") / "raw" / "creditcard.csv"
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / DATASET_RELATIVE_PATH).is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_PATH = PROJECT_ROOT / DATASET_RELATIVE_PATH

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. Place creditcard.csv in data/raw/ "
        "and rerun this notebook."
    )

raw_file_signature = (DATASET_PATH.stat().st_size, DATASET_PATH.stat().st_mtime_ns)
df = pd.read_csv(DATASET_PATH, low_memory=False)

required_columns = {"Time", "Amount", "Class"}
missing_required = required_columns.difference(df.columns)
if missing_required:
    raise ValueError(f"Expected columns are missing: {sorted(missing_required)}")
if df.empty:
    raise ValueError("The fraud dataset contains no transaction rows.")

class_labels = {0: "Legitimate", 1: "Fraud"}
unknown_class_values = set(df["Class"].dropna().unique()).difference(class_labels)
if unknown_class_values:
    raise ValueError(f"Unexpected Class values: {sorted(unknown_class_values)}")

print(f"Dataset: {DATASET_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Missing cells: {int(df.isna().sum().sum()):,}")
print("Column data types:")
display(df.dtypes.rename("dtype").to_frame())
print("Missing values by column:")
display(df.isna().sum().rename("missing_count").to_frame())
display(df.head())

Dataset: c:\Users\Chaithanya N V\FinGuard-AI\data\raw\creditcard.csv
Shape: 284,807 rows x 31 columns
Missing cells: 0
Column data types:


,dtype
Time,float64
V1,float64
V2,float64
V3,float64
V4,float64
V5,float64
V6,float64
V7,float64
V8,float64
V9,float64


Missing values by column:


,missing_count
Time,0
V1,0
V2,0
V3,0
V4,0
V5,0
V6,0
V7,0
V8,0
V9,0


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 2. Legitimate vs Fraudulent Transaction Counts

The count chart shows the actual number of transactions in each class. A logarithmic count axis keeps the small fraud class visible beside the much larger legitimate class; every row is included.

In [2]:
class_counts = df["Class"].value_counts().reindex([0, 1], fill_value=0)
class_count_table = pd.DataFrame(
    {
        "class": [class_labels[class_id] for class_id in class_counts.index],
        "transactions": class_counts.to_numpy(),
    }
)
display(class_count_table)

fig = px.bar(
    class_count_table,
    x="class",
    y="transactions",
    color="class",
    text="transactions",
    log_y=True,
    color_discrete_map={"Legitimate": "#356b8c", "Fraud": "#d85c41"},
    title="Transaction counts by class (logarithmic y-axis)",
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Transactions (log scale)", xaxis_title="")
fig.show()

,class,transactions
0,Legitimate,284315
1,Fraud,492


## 3. Class Imbalance Percentages

The percentage table gives the exact class prevalence. The chart uses a logarithmic axis only to make the rare class legible; compare its labeled values rather than interpreting bar lengths as a linear ratio. This extreme imbalance means raw accuracy can be misleading, so later model evaluation should emphasize minority-class precision/recall and precision-recall metrics.

In [3]:
class_percentages = (class_counts / len(df) * 100).rename("percentage")
class_percentage_table = pd.DataFrame(
    {
        "class": [class_labels[class_id] for class_id in class_percentages.index],
        "transactions": class_counts.to_numpy(),
        "percentage": class_percentages.to_numpy(),
    }
)
display(class_percentage_table.style.format({"percentage": "{:.4f}%"}))

fig = px.bar(
    class_percentage_table,
    x="class",
    y="percentage",
    color="class",
    text=class_percentage_table["percentage"].map(lambda value: f"{value:.4f}%"),
    log_y=True,
    color_discrete_map={"Legitimate": "#356b8c", "Fraud": "#d85c41"},
    title="Class share (logarithmic percentage axis)",
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Share of all transactions (%)", xaxis_title="")
fig.show()

,class,transactions,percentage
0,Legitimate,284315,99.8273%
1,Fraud,492,0.1727%


## 4. Transaction Amount Distributions

This overlaid histogram compares the full legitimate and fraud amount distributions after a `log1p` display transform. Each class is normalized to a probability density, so the legitimate majority does not overwhelm the fraud shape. The transform is only for visualization; original `Amount` values are preserved.

In [4]:
amount_log1p = np.log1p(df["Amount"].clip(lower=0))
amount_histogram = go.Figure()
for class_id, label, color in ((0, "Legitimate", "#356b8c"), (1, "Fraud", "#d85c41")):
    class_values = amount_log1p.loc[df["Class"].eq(class_id)]
    amount_histogram.add_trace(
        go.Histogram(
            x=class_values,
            name=label,
            histnorm="probability density",
            nbinsx=60,
            opacity=0.62,
            marker_color=color,
        )
    )
amount_histogram.update_layout(
    barmode="overlay",
    title="Transaction amount distribution by class (log1p Amount)",
    xaxis_title="log(1 + Amount)",
    yaxis_title="Probability density within class",
)
amount_histogram.show()

## 5. Fraud Amount vs Legitimate Amount Comparisons

The paired panels compare total transaction value and mean value per transaction. They use separate y-axes because totals and per-transaction averages have different scales. Totals reflect each class's count as well as its amounts; the mean panel compares average transaction size directly.

In [5]:
amount_summary = (
    df.assign(class_name=df["Class"].map(class_labels))
    .groupby("class_name", observed=True)["Amount"]
    .agg(transaction_count="size", total_amount="sum", mean_amount="mean", median_amount="median")
    .reindex(["Legitimate", "Fraud"])
)
display(amount_summary.style.format("{:.2f}", subset=["total_amount", "mean_amount", "median_amount"]))

amount_comparison = amount_summary.reset_index()
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Total transaction amount", "Mean amount per transaction"),
)
fig.add_trace(
    go.Bar(
        x=amount_comparison["class_name"],
        y=amount_comparison["total_amount"],
        text=amount_comparison["total_amount"].map(lambda value: f"{value:,.2f}"),
        textposition="outside",
        marker_color=["#356b8c", "#d85c41"],
        name="Total amount",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=amount_comparison["class_name"],
        y=amount_comparison["mean_amount"],
        text=amount_comparison["mean_amount"].map(lambda value: f"{value:,.2f}"),
        textposition="outside",
        marker_color=["#356b8c", "#d85c41"],
        name="Mean amount",
    ),
    row=1,
    col=2,
)
fig.update_layout(title="Transaction amount comparison", showlegend=False)
fig.update_yaxes(title_text="Total amount", row=1, col=1)
fig.update_yaxes(title_text="Mean amount", row=1, col=2)
fig.show()

,transaction_count,total_amount,mean_amount,median_amount
class_name,,,,
Legitimate,284315,25102462.04,88.29,22.00
Fraud,492,60127.97,122.21,9.25


## 6. Transaction Time Distributions

`Time` is elapsed seconds from the first recorded transaction. Each class is normalized independently to percent per time bin, allowing comparison of temporal shape without letting the legitimate majority dominate. The chart retains all records.

In [6]:
time_histogram = go.Figure()
for class_id, label, color in ((0, "Legitimate", "#356b8c"), (1, "Fraud", "#d85c41")):
    time_histogram.add_trace(
        go.Histogram(
            x=df.loc[df["Class"].eq(class_id), "Time"],
            name=label,
            histnorm="percent",
            nbinsx=48,
            opacity=0.58,
            marker_color=color,
        )
    )
time_histogram.update_layout(
    barmode="overlay",
    title="Transaction time distribution by class",
    xaxis_title="Elapsed time (seconds)",
    yaxis_title="Percent of transactions within class per bin",
)
time_histogram.show()

print("Observed elapsed-time range by class:")
display(
    df.assign(class_name=df["Class"].map(class_labels))
    .groupby("class_name", observed=True)["Time"]
    .agg(minimum_seconds="min", median_seconds="median", maximum_seconds="max")
)

Observed elapsed-time range by class:


,minimum_seconds,median_seconds,maximum_seconds
class_name,,,
Fraud,406.0,75568.5,170348.0
Legitimate,0.0,84711.0,172792.0


## 7. Correlation Analysis

The heatmap shows Pearson correlations among all numeric features, including `Class`. The accompanying ranking lists the strongest absolute feature-to-Class correlations. These are univariate linear associations; the anonymized `V` columns are PCA components, and correlation does not imply causation or establish predictive performance.

In [7]:
numeric_frame = df.select_dtypes(include="number")
correlation_matrix = numeric_frame.corr(method="pearson")
correlation_figure = px.imshow(
    correlation_matrix,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Pearson correlation matrix for all numeric features",
    labels={"x": "Feature", "y": "Feature", "color": "Correlation"},
)
correlation_figure.update_layout(height=900)
correlation_figure.show()

class_correlations = (
    correlation_matrix["Class"]
    .drop(labels=["Class"])
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
strongest_class_correlations = class_correlations.head(10).rename("pearson_correlation").to_frame()
display(strongest_class_correlations)

correlation_bar = px.bar(
    strongest_class_correlations.reset_index(names="feature"),
    x="pearson_correlation",
    y="feature",
    orientation="h",
    color="pearson_correlation",
    color_continuous_scale="RdBu_r",
    range_color=(-1, 1),
    title="Ten features with strongest absolute correlation to Class",
)
correlation_bar.update_layout(yaxis={"categoryorder": "total ascending"})
correlation_bar.show()

,pearson_correlation
V17,-0.326481
V14,-0.302544
V12,-0.260593
V10,-0.216883
V16,-0.196539
V3,-0.192961
V7,-0.187257
V11,0.154876
V4,0.133447
V18,-0.111485


## 8. Histograms of Key Features

These panels compare `V1` through `V5` and the visualization-only `log1p(Amount)` transform. Each class trace is normalized to its own density, so the rare fraud class remains visible. The `V` columns are anonymized PCA features; these shapes are descriptive, not direct business explanations.

In [8]:
histogram_features = ["V1", "V2", "V3", "V4", "V5", "Amount"]
missing_histogram_features = set(histogram_features).difference(df.columns)
if missing_histogram_features:
    raise ValueError(f"Expected histogram features are missing: {sorted(missing_histogram_features)}")

feature_histograms = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=["log1p(Amount)" if feature == "Amount" else feature for feature in histogram_features],
)
for feature_index, feature in enumerate(histogram_features):
    row, column = divmod(feature_index, 3)
    row += 1
    column += 1
    for class_id, label, color in ((0, "Legitimate", "#356b8c"), (1, "Fraud", "#d85c41")):
        raw_values = df.loc[df["Class"].eq(class_id), feature]
        values = np.log1p(raw_values.clip(lower=0)) if feature == "Amount" else raw_values
        feature_histograms.add_trace(
            go.Histogram(
                x=values,
                name=label,
                legendgroup=label,
                showlegend=feature_index == 0,
                histnorm="probability density",
                nbinsx=50,
                opacity=0.56,
                marker_color=color,
            ),
            row=row,
            col=column,
        )
feature_histograms.update_layout(
    barmode="overlay",
    title="Class-normalized histograms of selected features",
    height=720,
)
feature_histograms.show()

## 9. Boxplots for Outlier Detection

The boxplots show class-specific quartiles and 1.5-IQR whiskers for `Amount` (on `log1p` scale), `V1`–`V5`, and `Time`. Outlier counts are calculated against those whiskers for the full data. Individual points are omitted to keep the figure readable; no observations are removed.

In [9]:
boxplot_features = ["Amount", "V1", "V2", "V3", "V4", "V5", "Time"]
boxplot_figure = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=["log1p(Amount)" if feature == "Amount" else feature for feature in boxplot_features],
)
outlier_summaries = []
for feature_index, feature in enumerate(boxplot_features):
    row, column = divmod(feature_index, 3)
    row += 1
    column += 1
    for class_id, label, color in ((0, "Legitimate", "#356b8c"), (1, "Fraud", "#d85c41")):
        raw_values = df.loc[df["Class"].eq(class_id), feature].dropna().to_numpy()
        values = np.log1p(np.clip(raw_values, a_min=0, a_max=None)) if feature == "Amount" else raw_values
        q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75])
        iqr = q3 - q1
        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr
        outlier_count = int(((values < lower_fence) | (values > upper_fence)).sum())
        outlier_summaries.append(
            {"feature": feature, "class": label, "outside_iqr_whiskers": outlier_count}
        )
        boxplot_figure.add_trace(
            go.Box(
                x=[label],
                q1=[q1],
                median=[median],
                q3=[q3],
                lowerfence=[lower_fence],
                upperfence=[upper_fence],
                name=label,
                legendgroup=label,
                showlegend=feature_index == 0,
                marker_color=color,
                boxpoints=False,
            ),
            row=row,
            col=column,
        )
boxplot_figure.update_layout(
    title="Class-specific boxplots with 1.5-IQR whiskers",
    height=900,
    boxmode="group",
)
boxplot_figure.show()
display(pd.DataFrame(outlier_summaries))

,feature,class,outside_iqr_whiskers
0,Amount,Legitimate,255
1,Amount,Fraud,0
2,V1,Legitimate,6912
3,V1,Fraud,52
4,V2,Legitimate,13327
5,V2,Fraud,46
6,V3,Legitimate,3095
7,V3,Fraud,53
8,V4,Legitimate,10918
9,V4,Fraud,0


## 10. Fraud Amount Analysis

This section summarizes only rows whose `Class` is fraud, without resampling or filtering them further. The violin and embedded box show the distribution of `log1p(Amount)` across all fraudulent transactions, while the table reports amounts in their original units.

In [10]:
fraud_amounts = df.loc[df["Class"].eq(1), "Amount"].dropna()
if fraud_amounts.empty:
    raise ValueError("No fraudulent transactions are available for fraud amount analysis.")

fraud_amount_statistics = fraud_amounts.describe(percentiles=[0.25, 0.5, 0.75]).to_frame("fraud_amount")
fraud_amount_statistics.loc["total"] = fraud_amounts.sum()
display(fraud_amount_statistics.style.format("{:,.2f}"))

fraud_amount_plot = pd.DataFrame(
    {"log1p_amount": np.log1p(fraud_amounts.clip(lower=0)), "class": "Fraud"}
)
fraud_violin = px.violin(
    fraud_amount_plot,
    x="log1p_amount",
    y="class",
    orientation="h",
    box=True,
    points="all",
    color="class",
    color_discrete_map={"Fraud": "#d85c41"},
    title="Fraud transaction amounts (all fraud records, log1p scale)",
)
fraud_violin.update_layout(xaxis_title="log(1 + Amount)", yaxis_title="")
fraud_violin.show()

,fraud_amount
count,492.00
mean,122.21
std,256.68
min,0.00
25%,1.00
50%,9.25
75%,105.89
max,"2,125.87"
total,"60,127.97"


## 11. Insights Based on Actual Dataset Results

The statements below are calculated from this run's data. They describe this dataset only and do not claim model performance or causal relationships.

In [11]:
legitimate_count = int(class_counts.loc[0])
fraud_count = int(class_counts.loc[1])
fraud_percentage = float(class_percentages.loc[1])
legitimate_amount = amount_summary.loc["Legitimate"]
fraud_amount = amount_summary.loc["Fraud"]
strongest_feature = str(class_correlations.index[0])
strongest_correlation = float(class_correlations.iloc[0])
fraud_time = df.loc[df["Class"].eq(1), "Time"]

print(f"Transactions analyzed: {len(df):,} across {df.shape[1]} columns.")
print(
    f"Class balance: {legitimate_count:,} legitimate ({class_percentages.loc[0]:.4f}%) and "
    f"{fraud_count:,} fraud ({fraud_percentage:.4f}%)."
)
print(
    f"The fraud class is approximately 1 in {len(df) / fraud_count:,.1f} transactions; "
    "accuracy alone would not describe minority-class performance well."
)
print(
    f"Mean Amount: legitimate {legitimate_amount['mean_amount']:.2f}; "
    f"fraud {fraud_amount['mean_amount']:.2f}."
)
print(
    f"Median Amount: legitimate {legitimate_amount['median_amount']:.2f}; "
    f"fraud {fraud_amount['median_amount']:.2f}."
)
print(
    f"Fraud Amount total: {fraud_amounts.sum():,.2f}; "
    f"median: {fraud_amounts.median():,.2f}; "
    f"90th percentile: {fraud_amounts.quantile(0.90):,.2f}."
)
print(
    f"Fraud elapsed-time range: {fraud_time.min():,.0f} to {fraud_time.max():,.0f} seconds "
    f"({(fraud_time.max() - fraud_time.min()) / 3600:.2f} hours)."
)
print(
    f"Strongest absolute Pearson correlation with Class: {strongest_feature} "
    f"({strongest_correlation:.4f}). This is association, not causation."
)
print("The `V1`-`V28` columns are anonymized PCA features; their values are not directly interpretable as original transaction attributes.")

current_raw_signature = (DATASET_PATH.stat().st_size, DATASET_PATH.stat().st_mtime_ns)
assert current_raw_signature == raw_file_signature, "The raw CSV changed while the notebook was running."
assert int(df["Class"].eq(1).sum()) == fraud_count, "Fraud records were lost during analysis."
print("Validation passed: all fraud rows remain included, and the raw CSV signature is unchanged.")

Transactions analyzed: 284,807 across 31 columns.
Class balance: 284,315 legitimate (99.8273%) and 492 fraud (0.1727%).
The fraud class is approximately 1 in 578.9 transactions; accuracy alone would not describe minority-class performance well.
Mean Amount: legitimate 88.29; fraud 122.21.
Median Amount: legitimate 22.00; fraud 9.25.
Fraud Amount total: 60,127.97; median: 9.25; 90th percentile: 346.75.
Fraud elapsed-time range: 406 to 170,348 seconds (47.21 hours).
Strongest absolute Pearson correlation with Class: V17 (-0.3265). This is association, not causation.
The `V1`-`V28` columns are anonymized PCA features; their values are not directly interpretable as original transaction attributes.
Validation passed: all fraud rows remain included, and the raw CSV signature is unchanged.
